In [1]:
# Day_008_RNN_LSTM/RNN_LSTM_Pro.py
# -*- coding: utf-8 -*-
"""
نسخه حرفه‌ای‌تر RNN و LSTM با PyTorch
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ====== داده مصنوعی ======
# 100 نمونه، هر نمونه 10 زمان گام، 1 ویژگی
torch.manual_seed(42)
X = torch.randn(100, 10, 1)
y = torch.randint(0, 2, (100, 1), dtype=torch.float)

# Split train/test
train_size = 80
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# ====== مدل RNN ======
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNClassifier, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]  # خروجی آخرین زمان گام
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

# ====== مدل LSTM ======
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

# ====== پارامترها ======
input_size = 1
hidden_size = 16
output_size = 1
epochs = 20

rnn_model = RNNClassifier(input_size, hidden_size, output_size)
lstm_model = LSTMClassifier(input_size, hidden_size, output_size)

criterion = nn.BCELoss()
optimizer_rnn = torch.optim.Adam(rnn_model.parameters(), lr=0.01)
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=0.01)

# ====== تابع آموزش ======
def train(model, optimizer, loader):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

# ====== تابع ارزیابی ======
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            pred = (out > 0.5).float()
            correct += (pred == yb).sum().item()
    return correct / len(loader.dataset)

# ====== آموزش ======
for epoch in range(epochs):
    train_loss_rnn = train(rnn_model, optimizer_rnn, train_loader)
    train_loss_lstm = train(lstm_model, optimizer_lstm, train_loader)
    
    acc_rnn = evaluate(rnn_model, test_loader)
    acc_lstm = evaluate(lstm_model, test_loader)
    
    print(f"Epoch {epoch+1}/{epochs}: "
          f"RNN Loss={train_loss_rnn:.4f} Acc={acc_rnn:.2f}, "
          f"LSTM Loss={train_loss_lstm:.4f} Acc={acc_lstm:.2f}")

print("✅ آموزش RNN و LSTM به پایان رسید.")


Epoch 1/20: RNN Loss=0.6902 Acc=0.55, LSTM Loss=0.6715 Acc=0.50
Epoch 2/20: RNN Loss=0.6650 Acc=0.50, LSTM Loss=0.6647 Acc=0.50
Epoch 3/20: RNN Loss=0.6708 Acc=0.50, LSTM Loss=0.6640 Acc=0.50
Epoch 4/20: RNN Loss=0.6510 Acc=0.50, LSTM Loss=0.6570 Acc=0.50
Epoch 5/20: RNN Loss=0.6459 Acc=0.30, LSTM Loss=0.6522 Acc=0.50
Epoch 6/20: RNN Loss=0.6359 Acc=0.30, LSTM Loss=0.6330 Acc=0.50
Epoch 7/20: RNN Loss=0.6267 Acc=0.45, LSTM Loss=0.6136 Acc=0.40
Epoch 8/20: RNN Loss=0.6088 Acc=0.50, LSTM Loss=0.6048 Acc=0.45
Epoch 9/20: RNN Loss=0.5937 Acc=0.50, LSTM Loss=0.5885 Acc=0.50
Epoch 10/20: RNN Loss=0.5829 Acc=0.55, LSTM Loss=0.5811 Acc=0.45
Epoch 11/20: RNN Loss=0.5686 Acc=0.45, LSTM Loss=0.5896 Acc=0.50
Epoch 12/20: RNN Loss=0.5524 Acc=0.45, LSTM Loss=0.5843 Acc=0.45
Epoch 13/20: RNN Loss=0.5848 Acc=0.35, LSTM Loss=0.5732 Acc=0.45
Epoch 14/20: RNN Loss=0.5523 Acc=0.50, LSTM Loss=0.5788 Acc=0.50
Epoch 15/20: RNN Loss=0.5422 Acc=0.35, LSTM Loss=0.5703 Acc=0.45
Epoch 16/20: RNN Loss=0.5296 Acc=0